# LlamaFactory LoRA 全流程实战向导：微调 · 推理 · 测试集预测 · 评估 · 权重合并

本 Notebook 以**命令行示意**为主，完整覆盖从数据准备、LoRA 微调训练、模型推理对话、测试集预测、性能评估到权重合并导出的端到端全流程，适合作为 LlamaFactory 上手实践的一站式参考向导。

---
**硬件需求**：显存 ≥ 16GB（7B 模型 + LoRA + bf16），推荐 A100 / 3090 / 4090  
**框架版本**：LLaMA Factory ≥ 0.9.0

## 一、环境安装

### 1.1 安装 LLaMA Factory

克隆官方仓库并以可编辑模式安装，`[torch,metrics]` 会同时安装训练和评估所需的依赖：

```bash
git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
cd LLaMA-Factory
pip install -e ".[torch,metrics]"
```

### 1.2 验证安装

```bash
llamafactory-cli version
nvidia-smi
```

## 二、数据集准备

### 2.1 数据格式

LLaMA Factory 支持两种主流数据格式，按数据来源和对话轮次选择：

---

#### 格式一：Alpaca 格式（单轮，推荐用于指令微调）

适合**单轮问答**场景，每条样本由 instruction + input + output 三部分构成。  
保存到 `data/my_alpaca.json`：

```json
[
  {
    "instruction": "请将以下文本翻译成英文",
    "input": "人工智能正在改变世界。",
    "output": "Artificial intelligence is changing the world."
  },
  {
    "instruction": "写一首关于春天的五言绝句",
    "input": "",
    "output": "春风吹绿岸，燕子衔泥忙。花开千树艳，鸟鸣一声长。"
  }
]
```

| 字段 | 是否必填 | 说明 |
|---|---|---|
| `instruction` | 必填 | 任务指令，最终拼接为 prompt |
| `input` | 可为空字符串 | 用户的具体输入内容，为空时只用 instruction |
| `output` | 必填 | 期望的模型回答 |
| `system` | 可选 | 为该条样本单独指定 system prompt |

---

#### 格式二：ShareGPT 格式（多轮，推荐用于对话微调）

适合**多轮对话**场景，每条样本包含一段完整的对话历史。  
保存到 `data/my_sharegpt.json`：

```json
[
  {
    "system": "你是一个有帮助的中文AI助手。",
    "conversations": [
      {
        "from": "human",
        "value": "你好，请介绍一下大语言模型。"
      },
      {
        "from": "gpt",
        "value": "大语言模型（LLM）是基于 Transformer 架构、在海量文本上预训练的神经网络。"
      },
      {
        "from": "human",
        "value": "它和普通的神经网络有什么区别？"
      },
      {
        "from": "gpt",
        "value": "主要区别在于规模和预训练范式：LLM 参数量通常在 10亿以上，且通过自监督方式在互联网级语料上训练。"
      }
    ]
  }
]
```

| 字段 | 是否必填 | 说明 |
|---|---|---|
| `system` | 可选 | 全局 system prompt，适用于该条对话的所有轮次 |
| `conversations` | 必填 | 对话轮次列表，按时间顺序排列 |
| `conversations[i].from` | 必填 | 发言角色，`human` 表示用户，`gpt` 表示模型 |
| `conversations[i].value` | 必填 | 该轮次的具体文本内容 |

> **如何选择格式：**  
> - 数据来自人工标注的单轮问答 → Alpaca 格式  
> - 数据来自 ChatGPT / Claude 的多轮对话导出 → ShareGPT 格式  
> - 两种格式可以在 `dataset` 字段中混用，框架自动识别

### 2.2 注册数据集

编辑 `data/dataset_info.json`，两种格式的注册方式略有不同：

**Alpaca 格式注册**（通过 `columns` 映射字段名）：

```json
{
  "my_alpaca": {
    "file_name": "my_alpaca.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "system":   "system"
    }
  }
}
```

**ShareGPT 格式注册**（需额外声明 `formatting` 和 `tags` 定义角色标识）：

```json
{
  "my_sharegpt": {
    "file_name": "my_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations",
      "system":   "system"
    },
    "tags": {
      "role_tag":      "from",
      "content_tag":   "value",
      "user_tag":      "human",
      "assistant_tag": "gpt"
    }
  }
}
```

| 字段 | 说明 |
|---|---|
| `formatting` | Alpaca 格式省略此字段；ShareGPT 格式必须填 `"sharegpt"` |
| `columns.messages` | JSON 中对话列表对应的字段名（本例为 `conversations`） |
| `tags.role_tag` | 每条消息中标识角色的键名（本例为 `from`） |
| `tags.user_tag` | 用户角色的值（本例为 `human`） |
| `tags.assistant_tag` | 模型角色的值（本例为 `gpt`） |

---

**`file_name` 路径说明**

| 写法 | 起点 | 示例 | 适用场景 |
|---|---|---|---|
| 相对路径 | `dataset_info.json` 所在目录（默认 `data/`） | `"my_alpaca.json"` → `data/my_alpaca.json` | 数据集在 `data/` 目录下（推荐） |
| `..` 相对路径 | 同上，通过 `..` 向上跳出 `data/` | `"../raw/train.json"` → 项目根下的 `raw/train.json` | 可用，但路径深时难以维护 |
| **绝对路径** | **文件系统根目录，与 `data/` 完全无关** | `"/home/user/datasets/train.json"` | 数据集在 `data/` 之外时推荐使用 |

> 绝对路径**不需要经过 `data/` 目录**，直接从磁盘根（Linux: `/`，Windows: `C:\`）出发定位文件，框架检测到绝对路径后会跳过拼接直接加载。

---

注册后在训练配置的 `dataset` 字段中填入**键名**（即 `dataset_info.json` 中的 JSON key，不是文件名），**多个数据集可以用逗号拼接混用**：

```yaml
# 填写的是 dataset_info.json 中的键名（my_alpaca / my_sharegpt），而非文件名（my_alpaca.json）
dataset: my_alpaca,my_sharegpt   # 框架根据键名查找注册信息，自动合并并打乱顺序
```

### 2.3 思维链（CoT）数据集支持

DeepSeek-R1、Qwen3 等推理模型在回答前会先输出 `<think>...</think>` 包裹的推理过程，再给出最终答案。

LLaMA Factory 对思维链数据的处理方式：**将 `<think>...</think>` 和最终答案拼接后，整体写入 `output`（Alpaca）或 `gpt` 角色的 `value`（ShareGPT）字段**，框架将其作为普通目标序列进行 SFT 训练，无需任何额外配置。

---

#### Alpaca 格式

```json
[
  {
    "instruction": "计算 15 的阶乘",
    "input": "",
    "output": "<think>\n15! = 15 × 14 × ... × 1，逐步累乘得到结果。\n</think>\n\n15! = 1307674368000"
  },
  {
    "instruction": "判断 127 是否为质数",
    "input": "",
    "output": "<think>\n检验 2 到 √127 ≈ 11.3 之间的整数：2、3、5、7、11 均不能整除 127。\n</think>\n\n127 是质数。"
  }
]
```

`dataset_info.json` 注册与普通 Alpaca 数据集完全相同，**无需新增任何字段**：

```json
{
  "my_cot_alpaca": {
    "file_name": "my_cot_alpaca.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output"
    }
  }
}
```

---

#### ShareGPT 格式（多轮对话）

`<think>...</think>` 写在 `gpt` 角色的 `value` 中，与最终回答拼接在一起：

```json
[
  {
    "conversations": [
      {"from": "human", "value": "判断 127 是否为质数"},
      {"from": "gpt",   "value": "<think>\n检验 2 到 √127 ≈ 11.3 之间的整数均不能整除 127。\n</think>\n\n127 是质数。"}
    ]
  }
]
```

`dataset_info.json` 注册同样无需额外字段：

```json
{
  "my_cot_sharegpt": {
    "file_name": "my_cot_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations"
    }
  }
}
```

---

#### `enable_thinking` 参数（推理型模型专用训练配置）

对于具备推理能力的模型（如 Qwen3），当训练数据**不含** CoT 时，LLaMA Factory 提供 `enable_thinking` 参数自动处理；**数据已包含 `<think>...</think>` 时该参数不影响训练行为**。

| `enable_thinking` | 模式 | 行为 |
|---|---|---|
| `true`（默认） | 慢思考（Slow Thinking） | 自动在 `output` 前插入空 `<think></think>`，**计入 loss** |
| `false` | 快思考（Fast Thinking） | 自动在 prompt 末尾追加空 `<think></think>`，**不计入 loss** |
| `null` | 混合模式 | 含 CoT 的样本走慢思考，不含 CoT 的走快思考（配置较复杂，谨慎使用） |

对于同时提供推理版和非推理版的模型（如 Qwen3 系列），若只需训练**非推理模式**，使用 `_nothink` 后缀模板，可完全去掉 `<think>` token：

```yaml
# 方式一：完全不包含 <think> token
template: qwen3_nothink

# 方式二：使用官方 fast thinking 格式（与官方 enable_thinking=False 推理行为一致）
template: qwen3
enable_thinking: false
```

> **两者的区别**：`qwen3_nothink` 直接不生成 `<think>` 相关内容；`qwen3` + `enable_thinking: false` 会在 prompt 侧插入空 `<think></think>` 但不计入 loss，与官方推理时 `enable_thinking=False` 的行为严格对齐，迁移性更好。

---

> ⚠️ **对所有带推理链（CoT）能力的模型的特别说明**
>
> `enable_thinking` 参数并非 Qwen3 专属，LLaMA Factory 会对所有模型统一应用该逻辑。当使用默认值 `enable_thinking: true` 且训练数据**不含 CoT**（即无 `<think>...</think>` 内容）时，框架会自动在每条样本的 `output` 前拼接空的 `<think></think>` 并**计入 loss**，导致模型学习到"直接输出答案、不进行推理"的行为模式。
>
> **实测结果**：以 DeepSeekR1DistillLlama8B 为例，使用无 CoT 数据 + `enable_thinking: true`（默认）微调后，推理时输出中**不再出现 `<think>` 标签**，模型原有的推理链能力被抑制。该结论同样适用于其他具备推理能力的模型，例如：
> - **DeepSeek-R1 系列**：DeepSeek-R1、DeepSeekR1-Distill-Qwen/Llama 等蒸馏变体
> - **QwQ 系列**：QwQ-32B 等阿里巴巴推出的推理模型
> - **Marco-o1**：AIDC-AI 基于 Qwen2.5 蒸馏的开源推理模型
> - **Skywork-o1**：Skywork 团队基于 Llama 系列蒸馏的推理模型
>
> **建议**：
> - 若希望**保留**模型的推理能力，应在训练数据中包含 `<think>...</think>` 格式的 CoT 内容；
> - 若明确只需要**快速响应**（不需要推理链），则当前默认行为符合预期，无需修改。

### 2.4 多模态数据集支持

LLaMA Factory ≥ 0.9.0 支持**图像、视频、音频**三种模态的数据集，两种数据格式（Alpaca / ShareGPT）均可使用。多模态数据集需要搭配对应的**视觉语言模型**（VLM）使用，如 Qwen2-VL、InternVL、LLaVA、Llama 4 等。

---

#### 核心机制：占位符（Placeholder）

在文本字段中嵌入占位符，告知框架在该位置插入对应的媒体内容：

| 模态 | 占位符 | 数据字段 |
|---|---|---|
| 图像 | `<image>` | `images`（路径列表） |
| 视频 | `<video>` | `videos`（路径列表） |
| 音频 | `<audio>` | `audios`（路径列表） |

> **严格匹配规则**：文本中占位符的**数量**必须与媒体路径列表的**长度**完全一致，否则框架报错。

---

#### 图像数据集

**Alpaca 格式**（单轮，图像路径挂在顶层 `images` 字段，`instruction` 中用 `<image>` 标记插入位置）：

```json
[
  {
    "instruction": "<image>请描述这张图片的内容。",
    "input": "",
    "output": "图中是一片金色的麦田，远处有蓝天白云，画面宁静开阔。",
    "images": ["data/images/wheat_field.jpg"]
  },
  {
    "instruction": "比较这两张图片的风格差异：<image> 和 <image>",
    "input": "",
    "output": "左图为水墨画风格，笔触简练；右图为油画风格，色彩浓烈。",
    "images": ["data/images/ink_painting.jpg", "data/images/oil_painting.jpg"]
  }
]
```

在 `dataset_info.json` 中注册（增加 `columns.images` 映射）：

```json
{
  "my_image_alpaca": {
    "file_name": "my_image_alpaca.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "images":   "images"
    }
  }
}
```

---

**ShareGPT 格式**（多轮对话，`<image>` 占位符写在 `conversations` 的 `value` 中）：

```json
[
  {
    "conversations": [
      {"from": "human", "value": "<image>这张图里有什么动物？"},
      {"from": "gpt",   "value": "图中有一只金毛寻回犬，正在草地上玩耍。"},
      {"from": "human", "value": "它大概几岁？"},
      {"from": "gpt",   "value": "从毛色和体型判断，大约是 1～2 岁的幼犬。"}
    ],
    "images": ["data/images/golden_retriever.jpg"]
  }
]
```

在 `dataset_info.json` 中注册（增加 `columns.images` 映射）：

```json
{
  "my_image_sharegpt": {
    "file_name": "my_image_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations",
      "images":   "images"
    },
    "tags": {
      "role_tag":      "from",
      "content_tag":   "value",
      "user_tag":      "human",
      "assistant_tag": "gpt"
    }
  }
}
```

---

#### 视频数据集

结构与图像完全一致，将字段名和占位符替换为 `videos` / `<video>`：

```json
[
  {
    "instruction": "<video>请总结这段视频的主要内容。",
    "input": "",
    "output": "视频展示了一段延时摄影，记录了花朵从含苞到盛开的完整过程，历时约 24 小时。",
    "videos": ["data/videos/flower_timelapse.mp4"]
  }
]
```

`dataset_info.json` 注册：

```json
{
  "my_video_dataset": {
    "file_name": "my_video_dataset.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "videos":   "videos"
    }
  }
}
```

---

#### 音频数据集

同理，将字段名和占位符替换为 `audios` / `<audio>`：

```json
[
  {
    "instruction": "<audio>请转录这段音频，并判断说话人的情绪。",
    "input": "",
    "output": "转录文本：「今天真的太开心了，终于拿到 offer！」\n情绪判断：兴奋、喜悦。",
    "audios": ["data/audios/speech_sample.wav"]
  }
]
```

`dataset_info.json` 注册：

```json
{
  "my_audio_dataset": {
    "file_name": "my_audio_dataset.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "audios":   "audios"
    }
  }
}
```

---

#### 混合模态（图像 + 音频）

一条样本可以同时包含多种模态，各字段独立列举，占位符在文本中分别标记位置：

```json
[
  {
    "instruction": "根据图片 <image> 和对应的语音描述 <audio>，判断两者内容是否一致。",
    "input": "",
    "output": "一致。图片展示的是一只猫趴在键盘上，语音描述的内容与之相符。",
    "images": ["data/images/cat_keyboard.jpg"],
    "audios": ["data/audios/cat_description.wav"]
  }
]
```

`dataset_info.json` 注册（同时声明 `images` 和 `audios`）：

```json
{
  "my_mixed_dataset": {
    "file_name": "my_mixed_dataset.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "images":   "images",
      "audios":   "audios"
    }
  }
}
```

---

#### 媒体文件路径说明

| 路径写法 | 说明 |
|---|---|
| 相对路径（如 `"images/a.jpg"`） | 框架将其拼接到训练配置中 `media_dir` 的值之后；若未设置 `media_dir`，则相对于**命令执行时的工作目录**（不是 `dataset_info.json` 所在目录） |
| 绝对路径（如 `"/home/user/images/a.jpg"`） | 直接定位文件，与 `media_dir` 完全无关，跨服务器迁移时需注意路径一致性 |
| HuggingFace Hub 路径 | 数据集托管在 Hub 上时，框架自动下载媒体文件 |

> **`media_dir` 字段**：在训练 YAML 配置中指定 `media_dir: /path/to/media/root`，框架会自动将数据文件中的所有相对路径拼接到该根目录之后。**推荐做法**：将所有媒体文件集中放到一个目录，`media_dir` 指向该目录，数据 JSON 中只写文件名（如 `"a.jpg"`），迁移时只需改 `media_dir` 一处即可。

---

#### 支持的视觉语言模型（VLM）

多模态数据集必须搭配支持对应模态的模型，训练时 `template` 字段须填写对应模板：

| 模型系列 | 支持模态 | `template` |
|---|---|---|
| Qwen2-VL / Qwen2.5-VL / QVQ | 图像、视频 | `qwen2_vl` |
| Qwen2.5-Omni | 图像、视频、音频 | `qwen2_omni` |
| InternVL 2.5 / 3.x | 图像、视频 | `intern_vl` |
| LLaVA-1.5 | 图像 | `llava` |
| LLaVA-NeXT | 图像 | `llava_next` |
| Llama 4 | 图像 | `llama4` |
| Mllama（Llama 3.2 Vision） | 图像 | `mllama` |
| Gemma3 | 图像 | `gemma3` |

> **注意**：纯文本模型（如 LLaMA-3、Qwen2.5 等）不支持多模态数据集，若强行传入 `images` 字段会被框架忽略或报错。

---

#### 注意事项

1. **占位符数量严格匹配**：文本中有几个 `<image>`，`images` 列表就必须有几个路径，多或少都会报错。
2. **媒体文件格式**：图像支持 JPEG/PNG/WebP 等常见格式；视频支持 MP4/AVI 等；音频支持 WAV/MP3/FLAC 等，具体取决于所用模型的处理器（Processor）。
3. **显存需求大幅增加**：图像/视频会被编码为大量 token（如一张 1024×1024 图像约占 256～1024 个视觉 token），相比纯文本微调显存占用显著上升，建议开启 `gradient_checkpointing: true`。
4. **`cutoff_len` 需适当增大**：视觉 token 会占用序列长度配额，建议将 `cutoff_len` 设为 4096 及以上，避免大量样本被截断。

## 三、LoRA 微调训练

### 3.1 修改训练配置文件

训练配置文件位于 `configs/lora_LlamaFactory_train.yaml`，启动训练前修改以下**必填字段**：

| 字段 | 说明 |
|---|---|
| `model_name_or_path` | 基础模型本地路径或 HuggingFace Hub ID |
| `template` | 对话模板，需与模型一致（如 `qwen` / `llama3` / `glm4`），完整对照见 §4.1 注释 |
| `dataset` | 在 `dataset_info.json` 中注册的数据集名称 |
| `output_dir` | 训练结果（LoRA 权重 + 检查点）的保存路径 |

其余参数（LoRA rank/alpha、学习率、早停等）均已配有详细注释，按需调整。

### 3.2 启动训练

**单卡训练**

```bash
llamafactory-cli train configs/lora_LlamaFactory_train.yaml
```

**多卡分布式训练（DeepSpeed）**

```bash
# --num_gpus 指定 GPU 卡数；deepspeed 字段在 YAML 中指定 Zero 策略配置文件
deepspeed --num_gpus=4 \
    $(which llamafactory-cli) train configs/lora_LlamaFactory_train.yaml
```

在训练配置文件中取消注释 `deepspeed` 字段以启用对应的 Zero 优化策略：

```yaml
# Zero-2：优化器状态 + 梯度跨卡分片，模型参数每卡完整保留（适合大多数场景）
deepspeed: examples/deepspeed/ds_z2_config.json

# Zero-3：在 Zero-2 基础上进一步将模型参数也跨卡分片（适合超大模型显存不足时）
# deepspeed: examples/deepspeed/ds_z3_config.json
```

> `examples/deepspeed/` 目录为 LLaMA Factory 仓库内置的配置文件，克隆仓库后即可直接使用。

**训练完成后，`output_dir` 目录结构如下：**

```
output/lora_sft/
├── checkpoint-500/                  # 中间检查点
├── adapter_model.safetensors        # 最终 LoRA 权重（仅几十 MB）
├── adapter_config.json              # LoRA 结构配置
├── trainer_state.json               # 训练历史与最优检查点信息
└── training_loss.png                # 训练 / 验证损失曲线图
```

> **判断训练质量：** train_loss 和 eval_loss 同步下降 → 正常；eval_loss 反弹而 train_loss 继续下降 → 过拟合，减少 epoch 或增大 lora_dropout。

## 四、推理（Chat 模式）

### 4.1 准备推理配置文件

新建 `configs/lora_LlamaFactory_infer.yaml`，填入以下关键字段：

```yaml
model_name_or_path: /path/to/your/base/model   # 基础模型路径
adapter_name_or_path: ./output/lora_sft        # 训练输出的 LoRA 权重目录
finetuning_type: lora                          # 告知框架加载 LoRA 适配器
# template 须与训练时保持完全一致，常见模板名对应关系（完整列表见官方 README）：
#   Qwen / Qwen1.5 / Qwen2 / Qwen2.5          → qwen
#   Qwen3 / Qwen3.5（思考模式）                → qwen3 / qwen3_5
#   Qwen3 / Qwen3.5（非思考模式）              → qwen3_nothink / qwen3_5_nothink
#   LLaMA 3 / LLaMA 3.3                       → llama3
#   GLM-4 / GLM-Z1                            → glm4 / glmz1
#   InternLM 2 / InternLM 3                   → intern2
#   Mistral / Mixtral                         → mistral
#   DeepSeek (LLM/Code/MoE)                  → deepseek
#   DeepSeek-R1 (Distill)                     → deepseekr1
template: qwen                                 # 示例：Qwen / Qwen2 / Qwen2.5 系列

# 生成参数
max_new_tokens: 512
temperature: 0.7
top_p: 0.9
```

框架会自动在内存中将 LoRA 适配器合并到基础模型再进行推理，**无需提前合并权重**。

### 4.2 启动交互式对话

```bash
llamafactory-cli chat configs/lora_LlamaFactory_infer.yaml
```

运行后在终端直接输入问题，框架返回模型回答，输入 `exit` 或按 `Ctrl+C` 退出。

**也可以启动 Web UI 进行可视化对话：**

```bash
llamafactory-cli webui
# 打开浏览器访问 http://localhost:7860
# 在 Chat 标签页填入模型路径和 LoRA 路径后即可对话
```

## 五、基准评估（Eval 模式）

### 5.1 评估原理与支持的基准

`llamafactory-cli eval` 专门用于在**多选题形式的标准基准**上评测模型能力，与训练时的 `eval_loss` 不同：

| 对比维度 | 训练验证（`eval_loss`） | 基准评估（`llamafactory-cli eval`） |
|---|---|---|
| 触发时机 | 训练过程中周期性执行 | 训练结束后独立运行 |
| 评测方式 | 对 ground-truth token 计算交叉熵损失 | 让模型从 ABCD 四选项中选出正确答案，计算准确率（Accuracy） |
| 输出指标 | `eval_loss`（越低越好） | 各学科准确率 + 整体平均准确率（越高越好） |
| 适用场景 | 监控收敛、判断过拟合 | 横向对比、与论文结果比较、上报排行榜 |

---

**官方支持的评估基准（`task` 字段的合法值仅以下三个）**

| `task` 字段值 | 基准名称 | 语言 | 学科数 | 题目总数 | 标准 `n_shot` | 标准 `lang` |
|---|---|---|---|---|---|---|
| `ceval_validation` | C-Eval（验证集） | 中文 | 52 | 13,948 | 5 | `zh` |
| `cmmlu_test` | CMMLU（测试集） | 中文 | 67 | 11,528 | 5 | `zh` |
| `mmlu_test` | MMLU（测试集） | 英文 | 57 | 14,042 | 5 | `en` |

> **注意**：`task` 值的格式为 `{基准名}_{split}`，分割信息（`validation`/`test`）已编码在任务名中，配置文件中**没有单独的 `split` 字段**。`arc_challenge`、`hellaswag` 等其他基准名不是合法的 `task` 值，填写后框架会报错。

---

> **数据集下载方式（以 C-Eval 为例，存放到默认目录 `evaluation/`）**：
> ```bash
> # 方式一：HuggingFace CLI（需配置镜像或科学上网）
> # C-Eval 官方 Hub ID 为 ceval/ceval-exam
> huggingface-cli download ceval/ceval-exam --local-dir ./evaluation/ceval
>
> # 方式二：ModelScope（国内推荐，无需翻墙）
> pip install modelscope
> python -c "from modelscope import snapshot_download; snapshot_download('ceval/ceval-exam', cache_dir='./evaluation')"
> ```

### 5.2 修改评估配置文件

评估配置文件位于 `configs/lora_LlamaFactory_eval.yaml`，启动评估前修改以下**必填字段**：

| 字段 | 说明 |
|---|---|
| `model_name_or_path` | 基础模型本地路径或 HuggingFace Hub ID |
| `adapter_name_or_path` | 训练输出的 LoRA 权重目录（评估基础模型时注释掉此项和 `finetuning_type`） |
| `template` | **固定填 `fewshot`**，eval 命令专用评估模板，与训练时对话模板无关 |
| `task` | 评估任务，合法值：`ceval_validation` / `cmmlu_test` / `mmlu_test` |
| `task_dir` | 评估数据集的本地存放目录（默认 `evaluation`） |
| `lang` | 提示词语言（`zh` 对应中文基准，`en` 对应英文基准） |
| `save_dir` | 评估结果输出目录（**不能已存在**，否则框架直接报错） |

**评估 LoRA 微调模型 vs 评估基础模型**

```yaml
# ── 评估 LoRA 微调后的模型（保留这两行）────────────────────────
finetuning_type: lora
adapter_name_or_path: ./output/lora_sft

# ── 评估未微调的基础模型（注释掉上面两行）──────────────────────
# finetuning_type: lora
# adapter_name_or_path: ./output/lora_sft
```

> **建议**：分别对基础模型和微调后模型各跑一次评估（`save_dir` 使用不同路径），对比两份 `summary.json` 中的 `average_acc`，可量化验证微调带来的效果提升。

### 5.3 启动评估

```bash
llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
```

评估完成后，`save_dir` 目录下会生成以下文件：

```
output/eval_results/
├── results.json       # 每个学科的准确率明细（如 ceval 共 52 个学科）
└── summary.json       # 各大类学科准确率汇总 + 整体平均准确率
```

**`summary.json` 示例**（C-Eval 评估结果片段）：

```json
{
  "average_acc": 0.6134,
  "STEM": { "acc": 0.5821 },
  "Social Science": { "acc": 0.6892 },
  "Humanities": { "acc": 0.6453 },
  "Other": { "acc": 0.5971 }
}
```

**微调前后效果对比建议**

```bash
# 第一步：评估基础模型（注释掉 adapter_name_or_path 后运行）
llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
# 将结果目录改名备份
mv output/eval_results output/eval_results_base

# 第二步：评估 LoRA 微调后模型（恢复 adapter_name_or_path 后运行）
llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
# 结果保存到 output/eval_results

# 第三步：对比 summary.json 中的 average_acc
# 差值 = 微调后准确率 − 基础模型准确率（正值表示微调有效提升了基准性能）
```

> **注意事项**
> - 评估耗时较长（7B 模型在 C-Eval 上约需 30~60 分钟），建议在训练完成后单独运行
> - 若基准数据集语言与 `lang` 设置不一致，评估结果会偏低（如用英文 prompt 评测中文基准）
> - `n_shot=5` 与论文标准一致，改为 `0` 可测试零样本能力（通常比 5-shot 低 3~8 个百分点）

### 5.4 在自定义数据集上评估（NLG Evaluation）

`llamafactory-cli eval` **不支持自定义数据集**，它只能评测三个内置多选题基准。若要在自己的数据集上做评估，需使用**另一条路径**：`llamafactory-cli train` 配合 `do_predict: true`。

---

**两种评估方式对比**

| 对比维度 | `llamafactory-cli eval` | `llamafactory-cli train` + `do_predict: true` |
|---|---|---|
| 支持数据集 | 仅内置基准（MMLU / C-Eval / CMMLU） | **任意自定义数据集**（在 `dataset_info.json` 中注册即可） |
| 评估命令 | `llamafactory-cli eval` | `llamafactory-cli train`（调用 train 命令，但不做训练） |
| 任务类型 | 多选题（ABCD 选项） | 开放式生成（模型自由生成文本） |
| 输出指标 | 各学科准确率 + 整体平均准确率 | BLEU-4、ROUGE-1、ROUGE-2、ROUGE-L |
| 生成方式 | 对选项计算 log-prob，选概率最高者 | 调用 `model.generate()` 自回归生成完整回答 |
| 关键配置 | `task`、`n_shot`、`lang` | `do_predict: true`、`predict_with_generate: true` |
| 配置文件 | `configs/lora_LlamaFactory_eval.yaml` | `configs/lora_LlamaFactory_predict.yaml` |

---

**启动自定义数据集评估**

```bash
llamafactory-cli train configs/lora_LlamaFactory_predict.yaml
```

运行前需修改 `configs/lora_LlamaFactory_predict.yaml` 中的以下字段：

| 字段 | 说明 |
|---|---|
| `model_name_or_path` | 基础模型路径 |
| `adapter_name_or_path` | LoRA 权重目录（评估基础模型时注释掉） |
| `template` | 对话模板，须与训练时一致（如 `llama3` / `qwen`），完整对照见 §4.1 注释 |
| `eval_dataset` | 评估数据集键名（需在 `dataset_info.json` 中注册） |
| `dataset_dir` | `dataset_info.json` 所在目录 |
| `output_dir` | 结果输出目录 |
| `max_new_tokens` | 最大生成长度，建议与数据集平均回答长度对齐 |

---

**输出文件结构**

```
output/predict_results/
├── generated_predictions.jsonl   # 每条样本的模型输出（逐行 JSON，可人工审阅）
├── predict_results.json          # BLEU-4 / ROUGE-1 / ROUGE-2 / ROUGE-L 汇总得分
└── all_results.json              # 包含所有指标的完整结果
```

**`predict_results.json` 示例**：

```json
{
  "predict_bleu-4": 28.53,
  "predict_rouge-1": 52.41,
  "predict_rouge-2": 31.87,
  "predict_rouge-l": 48.92,
  "predict_runtime": 312.4,
  "predict_samples_per_second": 3.2
}
```

> **ROUGE-L** 是综合衡量生成质量最常用的指标（基于最长公共子序列 F1），越高越好；**BLEU-4** 侧重精确率，对短文本敏感。两个指标结合判断更全面。

## 六、合并 LoRA 权重

### 6.1 合并原理

LoRA 的低秩矩阵 $B \cdot A$ 乘以缩放系数后直接加回原始权重，得到与原模型结构完全相同的完整模型：

$$W_{merged} = W_{base} + \frac{\alpha}{r} \cdot B \cdot A$$

| | 合并前（LoRA 动态加载） | 合并后（完整模型） |
|---|---|---|
| 磁盘占用 | 基础模型 + 几十 MB | 与基础模型等大 |
| 推理延迟 | 略高（运行时合并） | 与原始模型相同 |
| 适用场景 | 训练调试、快速切换多个 LoRA | 生产部署、进一步量化 |

### 6.2 准备合并配置文件

新建 `configs/lora_LlamaFactory_merge.yaml`：

```yaml
model_name_or_path: /path/to/your/base/model   # 基础模型路径
adapter_name_or_path: ./output/lora_sft        # LoRA 权重目录
finetuning_type: lora
template: llama3

export_dir: ./output/merged_model              # 合并后完整模型的保存路径
export_size: 2                                 # 每个分片文件最大 2 GB
export_device: cuda                            # 合并运算设备（显存不足时改为 cpu）
export_legacy_format: false                    # false=保存为 safetensors（推荐）
```

### 6.3 执行合并

```bash
llamafactory-cli export configs/lora_LlamaFactory_merge.yaml
```

合并完成后，`export_dir` 目录结构如下，可直接作为标准 HuggingFace 模型使用：

```
output/merged_model/
├── model-00001-of-00003.safetensors   # 合并后完整权重（分片）
├── model-00002-of-00003.safetensors
├── model-00003-of-00003.safetensors
├── model.safetensors.index.json
├── config.json
├── tokenizer.json
└── tokenizer_config.json
```

**验证合并后模型可正常推理：**

```bash
# 将推理配置中的 model_name_or_path 改为合并后目录，去掉 adapter_name_or_path
llamafactory-cli chat configs/lora_LlamaFactory_merged_infer.yaml
```

## 附：完整工作流总结

```
原始数据（JSON）
   │
   ▼ 二、注册到 data/dataset_info.json
   │
   ▼ 三、修改 configs/lora_LlamaFactory_train.yaml
   │
   ▼ llamafactory-cli train configs/lora_LlamaFactory_train.yaml
   │
   └─→ output/lora_sft/
           ├── adapter_model.safetensors   ← LoRA 权重
           └── training_loss.png
   │
   ├─▶ 四、推理（LoRA 动态加载）
   │    llamafactory-cli chat configs/lora_LlamaFactory_infer.yaml
   │
   ├─▶ 五、评估（两种方式，按需选择）
   │
   │    ① 内置基准评估（MMLU / C-Eval / CMMLU，多选题准确率）
   │    llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
   │    └─→ output/eval_results/
   │            ├── results.json    ← 各学科准确率明细
   │            └── summary.json   ← 整体平均准确率（average_acc）
   │
   │    ② 自定义数据集评估（任意数据集，BLEU / ROUGE）
   │    llamafactory-cli train configs/lora_LlamaFactory_predict.yaml
   │    └─→ output/predict_results/
   │            ├── generated_predictions.jsonl  ← 逐条生成结果
   │            └── predict_results.json         ← BLEU-4 / ROUGE-L 得分
   │
   ▼ 六、llamafactory-cli export configs/lora_LlamaFactory_merge.yaml
   │
   └─→ output/merged_model/   ← 标准 HuggingFace 完整模型
```

**下一步建议**
- 使用 `vllm` 或 `llama.cpp` 对合并后的模型进行量化部署
- 在 C-Eval / MMLU 等基准上对比微调前后 `average_acc`，量化微调收益
- 尝试 DPO / KTO 阶段对齐微调，提升模型的指令跟随能力